In [1]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       673Mi       9.1Gi       3.0Mi       3.0Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [10]:
!git clone https://github.com/networkx/networkx.git
%cd networkx

!git checkout networkx-2.8.8

!pip install -e .

fatal: destination path 'networkx' already exists and is not an empty directory.
/content/networkx/networkx/networkx
HEAD is now at 9256ef670 Designate 2.8.8 release
Obtaining file:///content/networkx/networkx/networkx
  Preparing metadata (setup.py) ... done
  Attempting uninstall: networkx
    Found existing installation: networkx 2.8.8
    Uninstalling networkx-2.8.8:
      Successfully uninstalled networkx-2.8.8
  Running setup.py develop for networkx
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
mapclassify 2.10.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.


## Baseline Description

Repository: networkx/networkx  
Commit: 9256ef6

Workload: Betweenness Centrality using Brandes algorithm.

Why this workload:
Betweenness centrality is widely used in graph analytics and is computationally expensive (O(V * E)), making it a good candidate for performance optimization.

Discovery:
Identified as a slow path due to repeated BFS traversals over large graphs.

In [3]:
import networkx as nx

G = nx.erdos_renyi_graph(n=2000, p=0.01, seed=42)

# Reproducibility

We fix the random seed (`seed=42`) while generating the graph to ensure that the same input graph is used across all runs.

This guarantees that baseline and candidate implementations are evaluated on identical data, making the performance comparison fair and reproducible.

In [4]:
def run():
    return nx.betweenness_centrality(G)

In [5]:
import numpy as np

output = run()
np.save("baseline_output.npy", output)

In [6]:
import time
import numpy as np

def benchmark(fn, warmup=1, runs=3):
    for _ in range(warmup):
        fn()

    times = []
    for _ in range(runs):
        start = time.time()
        fn()
        times.append(time.time() - start)

    times = np.array(times)
    print("Times:", times)
    print("Median:", np.median(times))
    print("IQR:", np.percentile(times, 75) - np.percentile(times, 25))

benchmark(run)

Times: [24.38840318 23.71842194 24.61511374]
Median: 24.388403177261353
IQR: 0.4483458995819092


In [7]:
import networkx as nx
print("NetworkX version:", nx.__version__)

NetworkX version: 2.8.8


# Test Verification

The NetworkX library was successfully installed and its version is confirmed above.

Due to runtime constraints in Google Colab, the full test suite was not executed. However, the core functionality used in this workload was verified through successful execution.